In [1]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
    size_adjusted_power_comparison,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = True
_AUGMENTED_PARAM = 'Pi_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.025
_MC_SAMPLES = 100_000
_MC_ALPHA = 0.05
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83   0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [3]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [4]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [5]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [6]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: True
Augmented measurement equation: OutGap
Augmented coefficient: Pi_coef
Monte Carlo replications: 100000
Noise Covariance:
 [[0.302 0.    0.   ]
 [0.    0.42  0.   ]
 [0.    0.    0.019]]


In [7]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 100000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,3.222,0.227,0.010,0.001,100000,32690,0.327,0.001,0.324,0.330
1,Infl,2.184,0.324,0.008,0.001,100000,19094,0.191,0.001,0.189,0.193
2,Rate,1.090,0.485,0.005,0.001,100000,6046,0.060,0.001,0.059,0.062


In [8]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 100000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.12,2.984,0.52,0.000,0.009,0.001,100000,6224,0.062,0.001,0.061,0.064,3.0,200,4
1,cov_identity,5.92,988.580,0.00,0.002,1.217,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


In [9]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-1.038,-0.051,1.461,-0.728,0.424,0.008,0.005,0.0,0.000,0.003,0.001,0.0,100000,10859,0.109,0.001,0.107,0.111
1,OutGap,x,-0.263,-0.074,0.242,-1.044,0.360,0.010,0.001,0.0,0.000,0.003,0.001,0.0,100000,16758,0.168,0.001,0.165,0.170
2,OutGap,r,-1.381,-0.046,2.062,-0.651,0.446,0.007,0.006,0.0,0.001,0.003,0.001,0.0,100000,8977,0.090,0.001,0.088,0.092
3,Infl,Pi,-0.443,-0.017,1.737,-0.243,0.487,0.005,0.006,0.0,0.000,0.003,0.001,0.0,100000,5930,0.059,0.001,0.058,0.061
4,Infl,x,0.008,0.003,0.288,0.046,0.494,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5416,0.054,0.001,0.053,0.056
5,Infl,r,-0.357,-0.008,2.450,-0.117,0.493,0.005,0.008,0.0,0.001,0.003,0.001,0.0,100000,5508,0.055,0.001,0.054,0.057
6,Rate,Pi,-0.055,-0.012,0.326,-0.173,0.492,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5566,0.056,0.001,0.054,0.057
7,Rate,x,0.019,0.024,0.054,0.341,0.478,0.006,0.000,0.0,0.000,0.003,0.001,0.0,100000,6537,0.065,0.001,0.064,0.067
8,Rate,r,-0.019,-0.001,0.459,-0.018,0.496,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5466,0.055,0.001,0.053,0.056


In [10]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-2.158,-0.185,0.808,-2.659,0.041,0.037,0.002,0.0,0.000,0.003,0.000,0.0,100000,79166,0.792,0.001,0.789,0.794
1,OutGap,x,-0.363,-0.188,0.132,-2.705,0.037,0.038,0.000,0.0,0.000,0.003,0.000,0.0,100000,81130,0.811,0.001,0.809,0.814
0,OutGap,r,0.029,0.001,1.978,0.008,0.582,0.003,0.005,0.0,0.001,0.002,0.001,0.0,100000,1143,0.011,0.000,0.011,0.012
5,Infl,Pi,-0.241,-0.016,0.975,-0.230,0.492,0.005,0.003,0.0,0.000,0.003,0.001,0.0,100000,5610,0.056,0.001,0.055,0.058
4,Infl,x,-0.030,-0.010,0.160,-0.147,0.497,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5149,0.051,0.001,0.050,0.053
3,Infl,r,-0.092,-0.002,2.346,-0.029,0.502,0.005,0.007,0.0,0.001,0.003,0.001,0.0,100000,4893,0.049,0.001,0.048,0.050
8,Rate,Pi,0.023,0.008,0.183,0.110,0.501,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,4901,0.049,0.001,0.048,0.050
7,Rate,x,0.009,0.019,0.030,0.269,0.491,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,5644,0.056,0.001,0.055,0.058
6,Rate,r,-0.047,-0.005,0.439,-0.070,0.502,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,4975,0.050,0.001,0.048,0.051


In [11]:
print("Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"]).round(3)

Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.809,-2.847,-1.038,-1.038,0.0,0.0,0.0,0.003,0.002,0.005,0.005,0.0,0.0,0.0
1,OutGap,x,-0.001,-0.261,-0.263,-0.263,0.0,0.0,0.0,0.000,0.000,0.001,0.001,0.0,0.0,0.0
2,OutGap,r,-0.175,-1.206,-1.381,-1.381,0.0,0.0,0.0,0.004,0.003,0.006,0.006,0.0,0.0,0.0
3,Infl,Pi,-0.023,-0.420,-0.443,-0.443,-0.0,0.0,0.0,0.001,0.006,0.006,0.006,0.0,0.0,0.0
4,Infl,x,0.003,0.005,0.008,0.008,-0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
5,Infl,r,-0.018,-0.339,-0.357,-0.357,0.0,0.0,0.0,0.001,0.008,0.008,0.008,0.0,0.0,0.0
6,Rate,Pi,-0.000,-0.055,-0.055,-0.055,-0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
7,Rate,x,-0.000,0.020,0.019,0.019,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.0,0.0,0.0
8,Rate,r,-0.005,-0.014,-0.019,-0.019,0.0,0.0,0.0,0.000,0.002,0.001,0.001,0.0,0.0,0.0


In [12]:
print("Innovation decomposition on raw predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"]).round(3)

Innovation decomposition on raw predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.880,-4.038,-2.158,-2.158,-0.0,0.0,0.0,0.002,0.001,0.002,0.002,0.0,0.0,0.0
1,OutGap,x,0.253,-0.615,-0.363,-0.363,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.0,0.0,0.0
2,OutGap,r,-0.958,0.987,0.029,0.029,-0.0,0.0,0.0,0.005,0.003,0.005,0.005,0.0,0.0,0.0
3,Infl,Pi,-0.002,-0.239,-0.241,-0.241,-0.0,0.0,0.0,0.001,0.003,0.003,0.003,0.0,0.0,0.0
4,Infl,x,0.000,-0.030,-0.030,-0.030,-0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
5,Infl,r,-0.012,-0.080,-0.092,-0.092,0.0,0.0,0.0,0.001,0.007,0.007,0.007,0.0,0.0,0.0
6,Rate,Pi,0.000,0.023,0.023,0.023,-0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
7,Rate,x,0.000,0.009,0.009,0.009,-0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.0,0.0,0.0
8,Rate,r,-0.003,-0.043,-0.047,-0.047,0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.

In [ ]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()



## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [ ]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

In [ ]:
res_mle

## Serial Autocorrelation Tests for the Augmented Model

In [ ]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

In [ ]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))